In [ ]:
"""
03_nlp_tecnicas.py — Técnicas de Procesamiento de Lenguaje Natural (NLP)
=========================================================================

Este módulo explora técnicas fundamentales de NLP aplicadas al
dataset de películas:

1. Preprocesamiento de texto (tokenización, stopwords, stemming, lematización)
2. Representación vectorial (Bag of Words, TF-IDF)
3. Extracción de características (n-gramas, keywords por género)
4. Similitud de documentos (coseno, euclidiana)
5. Visualización (nubes de palabras por género)

Ejecutar:
    python 03_nlp_tecnicas.py
"""

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
from collections import Counter

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [ ]:
from wordcloud import WordCloud

In [ ]:
from utils import load_and_clean_data

In [ ]:
# =============================================================================
# Configuración
# =============================================================================
OUTPUT_DIR = "output"

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
})

In [ ]:
def ensure_output_dir():
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)

=============================================================================
PARTE 1: Preprocesamiento de Texto
=============================================================================

In [ ]:
def demonstrate_preprocessing():
    """
    Demuestra paso a paso las técnicas de preprocesamiento de texto NLP.

    Pipeline completo:
    1. Texto original
    2. Conversión a minúsculas
    3. Remoción de caracteres especiales
    4. Tokenización
    5. Remoción de stopwords
    6. Stemming (PorterStemmer y SnowballStemmer)
    7. Lematización (WordNetLemmatizer)
    """
    print("\n" + "=" * 70)
    print("📝 PARTE 1: PREPROCESAMIENTO DE TEXTO")
    print("=" * 70)

    # Texto de ejemplo (sinopsis de Toy Story)
    text = (
        "Led by Woody, Andy's toys live happily in his room until Andy's "
        "birthday brings Buzz Lightyear onto the scene. Afraid of losing "
        "his place in Andy's heart, Woody plots against Buzz. But when "
        "circumstances separate Buzz and Woody from their owner, the duo "
        "eventually learns to put aside their differences."
    )

    print(f"\n   📄 Texto original ({len(text)} caracteres):")
    print(f"   '{text[:100]}...'")

    # --- Paso 1: Minúsculas ---
    text_lower = text.lower()
    print(f"\n   1️⃣  Conversión a minúsculas:")
    print(f"   '{text_lower[:80]}...'")

    # --- Paso 2: Remoción de caracteres especiales ---
    text_clean = re.sub(r"[^a-záéíóúüñ\s]", "", text_lower)
    print(f"\n   2️⃣  Remoción de puntuación y caracteres especiales:")
    print(f"   '{text_clean[:80]}...'")

    # --- Paso 3: Tokenización ---
    tokens = word_tokenize(text_clean)
    print(f"\n   3️⃣  Tokenización ({len(tokens)} tokens):")
    print(f"   {tokens[:15]}...")

    # --- Paso 4: Remoción de stopwords ---
    stop_en = set(stopwords.words("english"))
    tokens_filtered = [t for t in tokens if t not in stop_en and len(t) > 1]
    print(f"\n   4️⃣  Sin stopwords ({len(tokens_filtered)} tokens, removidos {len(tokens) - len(tokens_filtered)}):")
    print(f"   {tokens_filtered[:15]}...")
    print(f"\n   📋 Ejemplos de stopwords removidas:")
    removed = [t for t in tokens if t in stop_en][:10]
    print(f"   {removed}")

    # --- Paso 5: Stemming ---
    porter = PorterStemmer()
    snowball = SnowballStemmer("english")

    stems_porter = [porter.stem(t) for t in tokens_filtered]
    stems_snowball = [snowball.stem(t) for t in tokens_filtered]

    print(f"\n   5️⃣  Stemming (reducir palabras a su raíz):")
    print(f"\n   {'Palabra':<20} {'Porter':<20} {'Snowball':<20}")
    print(f"   {'─' * 20} {'─' * 20} {'─' * 20}")
    for word, p, s in zip(tokens_filtered[:10], stems_porter[:10], stems_snowball[:10]):
        marker = " ←" if word != p else ""
        print(f"   {word:<20} {p:<20} {s:<20}{marker}")

    # --- Paso 6: Lematización ---
    lemmatizer = WordNetLemmatizer()
    lemmas = [lemmatizer.lemmatize(t) for t in tokens_filtered]

    print(f"\n   6️⃣  Lematización (forma base del diccionario):")
    print(f"\n   {'Palabra':<20} {'Stem (Porter)':<20} {'Lema':<20}")
    print(f"   {'─' * 20} {'─' * 20} {'─' * 20}")
    for word, stem, lemma in zip(tokens_filtered[:10], stems_porter[:10], lemmas[:10]):
        print(f"   {word:<20} {stem:<20} {lemma:<20}")

    print("""
    ┌──────────────────────────────────────────────────────────────────┐
    │  💡 STEMMING vs LEMATIZACIÓN:                                   │
    │                                                                  │
    │  Stemming: Corta sufijos mecánicamente (rápido, impreciso)       │
    │    "running" → "run", "studies" → "studi" ❌                    │
    │                                                                  │
    │  Lematización: Busca la forma base real (lento, preciso)         │
    │    "running" → "run", "studies" → "study" ✅                    │
    │                                                                  │
    │  Para sistemas de recomendación con TF-IDF, scikit-learn         │
    │  maneja la tokenización internamente, pero entender estos        │
    │  conceptos es fundamental para NLP.                              │
    └──────────────────────────────────────────────────────────────────┘
    """)

=============================================================================
PARTE 2: Representación Vectorial de Texto
=============================================================================

In [ ]:
def demonstrate_vectorization():
    """
    Demuestra y compara Bag of Words (CountVectorizer) vs TF-IDF.
    """
    print("\n" + "=" * 70)
    print("📊 PARTE 2: REPRESENTACIÓN VECTORIAL DE TEXTO")
    print("=" * 70)

    # Mini-corpus para demostración
    corpus = [
        "A toy comes to life in a boy's room",
        "A boy and his toy go on an adventure",
        "A dark knight fights crime in the city",
        "Crime and justice in a dark city",
        "A space adventure with robots and aliens",
    ]
    labels = ["Toy Story (sim.)", "Toy Adventure", "Dark Knight (sim.)", "Crime City", "Space Adventure"]

    print("\n   📚 Mini-corpus de ejemplo:")
    for i, (label, doc) in enumerate(zip(labels, corpus)):
        print(f"   {i + 1}. [{label}] '{doc}'")

    # --- Bag of Words ---
    print(f"\n   {'─' * 60}")
    print(f"   📦 BAG OF WORDS (CountVectorizer)")
    print(f"   {'─' * 60}")

    count_vec = CountVectorizer(stop_words="english")
    bow_matrix = count_vec.fit_transform(corpus)
    vocab = count_vec.get_feature_names_out()

    print(f"\n   Vocabulario ({len(vocab)} términos): {list(vocab)}")
    print(f"\n   Matriz BoW (filas=docs, cols=términos):")

    bow_df = pd.DataFrame(bow_matrix.toarray(), columns=vocab, index=labels)
    print(f"   {bow_df.to_string()}")

    # --- TF-IDF ---
    print(f"\n   {'─' * 60}")
    print(f"   📐 TF-IDF (TfidfVectorizer)")
    print(f"   {'─' * 60}")

    tfidf_vec = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf_vec.fit_transform(corpus)

    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray().round(3),
        columns=tfidf_vec.get_feature_names_out(),
        index=labels
    )
    print(f"\n   Matriz TF-IDF (valores ponderados):")
    print(f"   {tfidf_df.to_string()}")

    # --- Comparación de similitud ---
    print(f"\n   {'─' * 60}")
    print(f"   🔍 SIMILITUD COSENO: BoW vs TF-IDF")
    print(f"   {'─' * 60}")

    cos_bow = cosine_similarity(bow_matrix)
    cos_tfidf = cosine_similarity(tfidf_matrix)

    print(f"\n   Similitud Coseno (Bag of Words):")
    cos_bow_df = pd.DataFrame(cos_bow.round(3), columns=labels, index=labels)
    print(f"   {cos_bow_df.to_string()}")

    print(f"\n   Similitud Coseno (TF-IDF):")
    cos_tfidf_df = pd.DataFrame(cos_tfidf.round(3), columns=labels, index=labels)
    print(f"   {cos_tfidf_df.to_string()}")

    print("""
    ┌──────────────────────────────────────────────────────────────────┐
    │  💡 BoW vs TF-IDF:                                              │
    │                                                                  │
    │  BoW: Cuenta frecuencias brutas. "the", "a", "is" tienen        │
    │       el mismo peso que "robot", "adventure", "crime".           │
    │                                                                  │
    │  TF-IDF: Pondera por rareza. Palabras únicas de un documento    │
    │          reciben más peso que palabras comunes.                   │
    │                                                                  │
    │  → TF-IDF generalmente produce mejores recomendaciones           │
    │    porque distingue mejor el contenido específico.               │
    └──────────────────────────────────────────────────────────────────┘
    """)

    return bow_matrix, tfidf_matrix, count_vec, tfidf_vec, labels

=============================================================================
PARTE 3: Extracción de Características del Dataset Real
=============================================================================

In [ ]:
def extract_features(df):
    """
    Extrae y visualiza características del dataset de películas.

    - Keywords más frecuentes por género
    - N-gramas más comunes en overviews
    - Nubes de palabras por género
    """
    ensure_output_dir()
    print("\n" + "=" * 70)
    print("🔎 PARTE 3: EXTRACCIÓN DE CARACTERÍSTICAS")
    print("=" * 70)

    # --- Keywords más frecuentes por género ---
    print("\n   📋 Keywords más frecuentes por género:")

    genres_to_analyze = ["Action", "Comedy", "Drama", "Horror", "Science Fiction"]

    for genre in genres_to_analyze:
        genre_movies = df[df["genres_list"].apply(lambda x: genre in x)]
        all_keywords = genre_movies["keywords_list"].explode()
        top_keywords = all_keywords.value_counts().head(10)

        print(f"\n   🎭 {genre} ({len(genre_movies)} películas):")
        for kw, count in top_keywords.items():
            bar = "█" * min(int(count / top_keywords.max() * 20), 20)
            print(f"      {kw:<25} {count:4d} {bar}")

    # --- N-gramas más comunes ---
    print(f"\n   {'─' * 60}")
    print(f"   📊 N-GRAMAS MÁS COMUNES EN SINOPSIS")
    print(f"   {'─' * 60}")

    overviews = df[df["overview"] != ""]["overview"]

    for n, label in [(1, "Unigramas"), (2, "Bigramas"), (3, "Trigramas")]:
        vec = CountVectorizer(stop_words="english", ngram_range=(n, n), max_features=10)
        matrix = vec.fit_transform(overviews)
        freqs = zip(vec.get_feature_names_out(), matrix.sum(axis=0).A1)
        sorted_freqs = sorted(freqs, key=lambda x: x[1], reverse=True)

        print(f"\n   📝 Top 10 {label}:")
        for term, freq in sorted_freqs:
            bar = "█" * min(int(freq / sorted_freqs[0][1] * 25), 25)
            print(f"      {term:<30} {int(freq):5d} {bar}")

    # --- Nubes de palabras por género ---
    print(f"\n   {'─' * 60}")
    print(f"   ☁️  GENERANDO NUBES DE PALABRAS POR GÉNERO")
    print(f"   {'─' * 60}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    genres_for_cloud = ["Action", "Comedy", "Drama", "Horror", "Science Fiction", "Romance"]

    for ax, genre in zip(axes.flatten(), genres_for_cloud):
        genre_movies = df[df["genres_list"].apply(lambda x: genre in x)]
        text = " ".join(genre_movies["overview"].dropna())

        if not text.strip():
            ax.set_title(f"{genre} (sin datos)")
            ax.axis("off")
            continue

        wc = WordCloud(
            width=600,
            height=400,
            background_color="#1a1a2e",
            colormap="magma",
            max_words=80,
            stopwords=set(stopwords.words("english")),
            collocations=False,
        ).generate(text)

        ax.imshow(wc, interpolation="bilinear")
        ax.set_title(f"☁️ {genre}", fontsize=14, fontweight="bold")
        ax.axis("off")

    plt.suptitle("Nubes de Palabras por Género (basadas en sinopsis)", fontsize=16, y=1.02)
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/03_wordclouds_por_genero.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   💾 Guardado: {path}")

=============================================================================
PARTE 4: Similitud de Documentos — Demostración Práctica
=============================================================================

In [ ]:
def demonstrate_document_similarity(df, n_sample=1000):
    """
    Demuestra la similitud de documentos sobre sinopsis reales.

    Compara similitud coseno vs distancia euclidiana y muestra
    por qué la similitud coseno es preferida para textos.

    Parámetros:
        df (pd.DataFrame): DataFrame de películas
        n_sample (int): Número de películas a usar (para velocidad)
    """
    ensure_output_dir()
    print("\n" + "=" * 70)
    print("📐 PARTE 4: SIMILITUD DE DOCUMENTOS")
    print("=" * 70)

    print("""
    ┌──────────────────────────────────────────────────────────────────┐
    │  SIMILITUD COSENO vs DISTANCIA EUCLIDIANA                       │
    │                                                                  │
    │  Coseno: Mide el ÁNGULO entre vectores                           │
    │    cos(A,B) = (A·B) / (||A|| × ||B||)                           │
    │    Rango: [0, 1] (para TF-IDF, siempre positivo)                │
    │    → Invariante a la longitud del documento                      │
    │    → Ideal para textos de diferente tamaño                       │
    │                                                                  │
    │  Euclidiana: Mide la DISTANCIA absoluta                          │
    │    d(A,B) = √(Σ(ai - bi)²)                                      │
    │    Rango: [0, ∞)                                                 │
    │    → Sensible a la longitud del documento                        │
    │    → Documentos largos parecen "lejos" de los cortos             │
    │                                                                  │
    │  → Para NLP, COSENO es casi siempre la mejor opción.             │
    └──────────────────────────────────────────────────────────────────┘
    """)

    # Tomar una muestra para velocidad
    sample = df[df["overview"] != ""].head(n_sample).copy()
    sample = sample.reset_index(drop=True)

    # Vectorizar
    tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
    tfidf_matrix = tfidf.fit_transform(sample["overview"])

    # Elegir una película de referencia
    ref_title = "Toy Story"
    ref_matches = sample[sample["title"] == ref_title]
    if ref_matches.empty:
        ref_idx = 0
        ref_title = sample.iloc[0]["title"]
    else:
        ref_idx = ref_matches.index[0]

    print(f"   🎬 Película de referencia: '{ref_title}'")
    print(f"   📝 Sinopsis: '{sample.iloc[ref_idx]['overview'][:120]}...'")

    # Calcular similitud coseno
    cos_sims = cosine_similarity(tfidf_matrix[ref_idx:ref_idx + 1], tfidf_matrix).flatten()

    # Calcular distancia euclidiana
    euc_dists = euclidean_distances(tfidf_matrix[ref_idx:ref_idx + 1], tfidf_matrix).flatten()

    # Top 5 por similitud coseno
    top_cos_idx = cos_sims.argsort()[::-1][1:6]
    print(f"\n   🔍 Top 5 más similares (Similitud Coseno):")
    for rank, idx in enumerate(top_cos_idx, 1):
        print(f"   {rank}. {sample.iloc[idx]['title']:<35} "
              f"Coseno: {cos_sims[idx]:.4f} │ "
              f"Euclidiana: {euc_dists[idx]:.4f}")

    # Top 5 por distancia euclidiana (menor = más similar)
    top_euc_idx = euc_dists.argsort()[1:6]
    print(f"\n   🔍 Top 5 más similares (Distancia Euclidiana):")
    for rank, idx in enumerate(top_euc_idx, 1):
        print(f"   {rank}. {sample.iloc[idx]['title']:<35} "
              f"Coseno: {cos_sims[idx]:.4f} │ "
              f"Euclidiana: {euc_dists[idx]:.4f}")

    # Visualizar la comparación
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histograma de similitud coseno
    axes[0].hist(cos_sims, bins=50, color="#e94560", edgecolor="#1a1a2e", alpha=0.85)
    axes[0].set_title(f"Distribución de Similitud Coseno\n(vs '{ref_title}')")
    axes[0].set_xlabel("Similitud Coseno")
    axes[0].set_ylabel("Cantidad de películas")
    axes[0].axvline(cos_sims[top_cos_idx[0]], color="#53d8fb", linestyle="--",
                    label=f"Más similar: {cos_sims[top_cos_idx[0]]:.3f}")
    axes[0].legend()

    # Histograma de distancia euclidiana
    axes[1].hist(euc_dists, bins=50, color="#0f3460", edgecolor="#1a1a2e", alpha=0.85)
    axes[1].set_title(f"Distribución de Distancia Euclidiana\n(vs '{ref_title}')")
    axes[1].set_xlabel("Distancia Euclidiana")
    axes[1].set_ylabel("Cantidad de películas")
    axes[1].axvline(euc_dists[top_euc_idx[0]], color="#53d8fb", linestyle="--",
                    label=f"Más cercana: {euc_dists[top_euc_idx[0]]:.3f}")
    axes[1].legend()

    plt.tight_layout()
    path = f"{OUTPUT_DIR}/03_coseno_vs_euclidiana.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n   💾 Guardado: {path}")

    # Scatter: Coseno vs Euclidiana
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(cos_sims, euc_dists, alpha=0.3, s=5, color="#e94560")
    ax.set_xlabel("Similitud Coseno")
    ax.set_ylabel("Distancia Euclidiana")
    ax.set_title(f"Coseno vs Euclidiana (referencia: '{ref_title}')")

    # Marcar top 5 coseno
    for idx in top_cos_idx:
        ax.annotate(sample.iloc[idx]["title"][:20],
                    (cos_sims[idx], euc_dists[idx]),
                    fontsize=8, color="#53d8fb")

    plt.tight_layout()
    path2 = f"{OUTPUT_DIR}/03_coseno_vs_euclidiana_scatter.png"
    plt.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   💾 Guardado: {path2}")

=============================================================================
PARTE 5: Efecto de Parámetros en la Vectorización
=============================================================================

In [ ]:
def demonstrate_parameter_effects(df):
    """
    Muestra cómo diferentes parámetros del vectorizador
    afectan el vocabulario y las recomendaciones.
    """
    print("\n" + "=" * 70)
    print("⚙️  PARTE 5: EFECTO DE PARÁMETROS EN LA VECTORIZACIÓN")
    print("=" * 70)

    overviews = df[df["overview"] != ""]["overview"]

    configs = [
        {"name": "Default", "params": {"stop_words": "english"}},
        {"name": "Bigramas", "params": {"stop_words": "english", "ngram_range": (1, 2)}},
        {"name": "Max 1000 feat.", "params": {"stop_words": "english", "max_features": 1000}},
        {"name": "Min DF=5", "params": {"stop_words": "english", "min_df": 5}},
        {"name": "Max DF=0.5", "params": {"stop_words": "english", "max_df": 0.5}},
        {"name": "Sublinear TF", "params": {"stop_words": "english", "sublinear_tf": True}},
    ]

    print(f"\n   {'Configuración':<20} {'Vocabulario':>12} {'Sparsidad':>12} {'NNZ':>12}")
    print(f"   {'─' * 20} {'─' * 12} {'─' * 12} {'─' * 12}")

    for config in configs:
        tfidf = TfidfVectorizer(**config["params"])
        matrix = tfidf.fit_transform(overviews)
        vocab_size = len(tfidf.vocabulary_)
        sparsity = 1.0 - (matrix.nnz / (matrix.shape[0] * matrix.shape[1]))
        nnz = matrix.nnz

        print(f"   {config['name']:<20} {vocab_size:>12,} {sparsity:>11.4%} {nnz:>12,}")

    print("""
    ┌──────────────────────────────────────────────────────────────────┐
    │  💡 RECOMENDACIONES DE PARÁMETROS:                              │
    │                                                                  │
    │  • stop_words="english": SIEMPRE usar para textos en inglés     │
    │  • ngram_range=(1,2): Captura frases como "science fiction"     │
    │  • max_features: Limita memoria, útil para datasets grandes      │
    │  • min_df=2: Ignora términos muy raros (probablemente ruido)    │
    │  • max_df=0.95: Ignora términos demasiado comunes               │
    │  • sublinear_tf=True: log(1+tf) reduce impacto de repetición    │
    └──────────────────────────────────────────────────────────────────┘
    """)

=============================================================================
Ejecución principal
=============================================================================

In [ ]:
if __name__ == "__main__":
    # Parte 1: Preprocesamiento
    demonstrate_preprocessing()

    # Parte 2: Vectorización
    demonstrate_vectorization()

    # Cargar datos reales
    print("\n" + "=" * 70)
    print("📂 CARGANDO DATOS PARA ANÁLISIS REAL")
    print("=" * 70)
    df = load_and_clean_data(include_keywords=True)

    # Parte 3: Extracción de características
    extract_features(df)

    # Parte 4: Similitud de documentos
    demonstrate_document_similarity(df, n_sample=2000)

    # Parte 5: Efecto de parámetros
    demonstrate_parameter_effects(df)

    print("\n" + "=" * 70)
    print("✅ Módulo 03 completado. Revisa las gráficas en output/")
    print("=" * 70)